# Interpretación del perfil con el LLM (P1-a)
Una sola llamada al modelo. **Lo único que sale de tu máquina** es el Data Profile y los metadatos de las columnas derivadas (nombres, tipos, estadísticas): ningún valor de ningún log. La regex y los ejemplos del Inspector no se envían, ni la nota del analista sobre la zona.

El modelo **propone**; el código **dispone**: cada consulta se valida contra un gemelo vacío del dataset y lo que no pasa se descarta con un código y un motivo.

In [ ]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace

from dfir_copilot.agent.llm import LLMConfig, make_llm
from dfir_copilot.interpret import (SYSTEM_PROMPT, LangChainStructured, build_user_message, columns_for, estimate_tokens,
                                   interpret_profile, render_interpretation, result_to_dict, wire_schema)
from dfir_copilot.profiling.inspector import inspect_source

RUTA = "/workspace/data/raw/three_months.csv"
IDIOMA = "es"
ZONA = "America/Santiago"   # zona de las horas del archivo si no la traen; None si nadie la ha declarado
ESPERA_MAX_S = 180   # la respuesta es larga (en la prueba real, 5 446 tokens y 53 s): 60 s no alcanza con margen

%time borrador = inspect_source(RUTA, lang=IDIOMA, timezone=ZONA)
perfil, derivadas = borrador.profile, borrador.derived
marca = borrador.mapping["timestamp"]   # formato y zona: el modelo sabe cómo se leyeron las horas
columnas = columns_for(perfil, derivadas, marca)
print("estado del borrador:", borrador.status, "| filas:", f"{perfil.dataset.row_count:,}")
print("columnas consultables:", list(columnas))

## 1. Qué se enviaría
Revisa el contenido exacto antes de llamar. No hay valores de logs: solo nombres de campo, estadísticas y formas abstractas. Las columnas consultables son las de la **tabla** (`endpoint`, `timestamp_utc`...); los nombres canónicos del **perfilador** (`uri`, `timestamp`...) se usan solo al opinar sobre el mapeo.

In [ ]:
mensaje = build_user_message(perfil, derivadas, columnas, IDIOMA, marca)
print(f"mensaje de usuario: {len(mensaje):,} caracteres | entrada total estimada: unos "
      f"{estimate_tokens(SYSTEM_PROMPT, mensaje, wire_schema()):,} tokens (con prompt de sistema y esquema de respuesta)\n")
print(mensaje[:3000])

## 2. La llamada
Usa la configuración del `.env` (`LLM_PROVIDER`, `LLM_MODEL`...). La espera máxima se fija aquí, no en el `.env`.

In [ ]:
cfg = replace(LLMConfig.from_env(), timeout_s=ESPERA_MAX_S)
print("proveedor:", cfg.provider, "| modelo:", cfg.resolved_model, "| espera máx.:", cfg.timeout_s, "s")
llm = LangChainStructured(make_llm(cfg))

%time resultado = interpret_profile(llm, perfil, derived=derivadas, lang=IDIOMA, timestamp=marca)
print(render_interpretation(resultado, IDIOMA))

## 3. Cómo leerlo
* **Consultas aceptadas**: pasaron el validador, pero siguen siendo *hipótesis*. Cada una trae `Se refuta si`: ese es el criterio para descartarla.
* **Descartado por el validador**: lo que el modelo inventó (columnas que no existen, SQL no permitido). Un descarte frecuente es una señal para ajustar el prompt, no un fallo del análisis.
* **Preguntas para el analista**: lo que el esquema no puede saber. Respóndelas antes de fiarte de las conclusiones.

In [ ]:
from pathlib import Path
import json

SALIDA = Path("/workspace/data/interpretaciones/three_months_p1a.json")
SALIDA.parent.mkdir(parents=True, exist_ok=True)
SALIDA.write_text(json.dumps(result_to_dict(resultado), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("guardado en", SALIDA, "(sin el perfil enviado)")

## 4. Ejecutar lo aceptado sobre los datos reales (solo local)
El validador ve una tabla vacía: no sabe si una consulta fallará con valores reales. Aquí se ejecutan **en tu máquina** y se mide cuántas fallan. Los resultados con valores reales **no vuelven al modelo**.

Es exploración, no evidencia: pasa por un motor suelto, sin ledger. La ejecución registrada llega con el agente (P1-b). Requiere el caso ya ingerido con el notebook 12, con la misma zona que declaras aquí.

In [ ]:
from dfir_copilot.cases import CaseWorkspace
from dfir_copilot.interpret import run_accepted_queries, summarize_runs

CASO = "IDOR-INVOICES-2020Q4-TZ-SANTIAGO"   # el caso con la MISMA zona (notebook 12): tiene timestamp_local
ws = CaseWorkspace.open_or_create(CASO, root="/workspace/data/cases", analyst="eder", lang=IDIOMA)
if not ws.meta["dataset"]:
    print("El caso no tiene datos ingeridos: ejecuta antes el notebook 12.")
    corridas = []
else:
    corridas = run_accepted_queries(ws.engine(), resultado, max_rows=20)
    print(summarize_runs(corridas))
    for c in corridas:
        print(f"[{c.status:>8}] {c.priority:<6} {c.id}: {c.row_count} filas, {c.elapsed_ms} ms", c.error or "")

In [ ]:
import pandas as pd

for c in corridas:
    if c.status == "ok":
        print(f"\n=== {c.id} ({c.row_count} filas{', truncado' if c.truncated else ''}) ===")
        display(pd.DataFrame(c.rows, columns=list(c.columns)).head(10))